# 01 从一个波长算起：雪粒半径怎样改变反照率

积雪像元变暗，可能来自雪面积减小，也可能来自雪粒变粗。开始解混以前，我想先把后一个因素单独算清楚：**固定纯净球形雪和照明条件，只改变半径，哪些波段最敏感？** 先预测：50 μm 变成 1000 μm 后，可见光和 1 μm 附近是否会下降同样多？把预测记下，再用图检验。

这页先在 1.025 μm 逐项代入 Wiscombe–Warren（1980，WW80）半无限雪层闭式解，再重算一条光谱和整个网格。输入为官方 SNICAR 202105 库中的单次散射反照率 $\omega$ 和不对称因子 $g$；它们已经包含上游 Mie 散射计算。我们重算的是辐射传输这一步。`ss_alb` 保留 float64，`asm_prm` 存为 float32，计算统一提升至 float64。

第一段代码会打印散射输入的三个轴：散射输入：148 半径 × 480 波长 × 2 光学选项。角度轴在正演时才加入，得到四维输出。本页应出现手算表、单条光谱校核图、四种半径的双角度光谱图，以及太阳谱加权反照率图。来源、转换和逐文件哈希见 `data/demo/scattering_inputs.source.json`。

正演输出统一称为**反照率（albedo）**，表示给定入射方向的半球总反射能量；影像观测称为**反射率（reflectance）**。实景解混将前者作为后者的雪端元近似。SNICAR-AD 全称为 Snow, Ice, and Aerosol Radiative Model with Adding-Doubling；其 adding-doubling 解保留为独立交叉核对分支。

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from snowmix import teaching as t
from snowmix.optics import ww80_semi_infinite
optics = t.load_optical_inputs()
angles_deg = np.array([0, 60, 22], dtype=np.float64)
nr, nw, no = optics["ss_alb"].shape
print(f"散射输入：{nr} 半径 × {nw} 波长 × {no} 光学选项")
print("绘图标签：" + ("中文" if t.configure_plots() else "English（未安装支持的中文字体）"))
assert optics["ss_alb"].shape == optics["asm_prm"].shape == (148, 480, 2)
assert not np.isnan(optics["ss_alb"]).any()

## 先把一个数算明白

取半径 200 μm、Warren 1984、太阳天顶角 60°。散射参数网格最接近 1.03 μm 的中心是 **1.025 μm**，这里使用实际中心。$\omega$ 表示一次作用后仍被散射的能量比例；$g$ 越大，前向散射越强。入射角通过 $\mu_0=\cos\theta_0$ 进入模型。

先做 delta 缩放，去除很窄的前向峰：
$$g^*=\frac{g}{1+g},\qquad
\omega^*=\frac{(1-g^2)\omega}{1-g^2\omega}.$$
再定义 $a=1-\omega^*g^*$、$b=g^*/a$、$\xi=\sqrt{3a(1-\omega^*)}$、$P=2\xi/(3a)$。半无限雪层直接入射反照率为
$$\alpha_\infty=\frac{\omega^*}{1+P}\frac{1-b\xi\mu_0}{1+\xi\mu_0}.$$

下面逐行算中间量，随后才调用共享函数。输出是该波长的无量纲方向—半球反照率；四位小数打印为 **0.6567**（中间量表显示 0.656708）。函数与手写公式的 $10^{-12}$ 校核使用未舍入值。这个例子把“较强吸收导致较低反照率”的描述落实为可以检查的数值关系。

In [ ]:
ri = int(np.flatnonzero(optics["radii_um"] == 200)[0])
wi = int(np.argmin(abs(optics["wavelength_um"] - 1.03)))
omega = float(optics["ss_alb"][ri, wi, 0])
g = float(optics["asm_prm"][ri, wi, 0])
mu0 = float(np.cos(np.deg2rad(60.0)))
g_star = g / (1 + g)
omega_star = (1 - g**2) * omega / (1 - g**2 * omega)
a = 1 - omega_star * g_star
b = g_star / a
xi = np.sqrt(3 * a * (1 - omega_star))
p = 2 * xi / (3 * a)
alpha_manual = omega_star / (1 + p) * (1 - b * xi * mu0) / (1 + xi * mu0)
alpha_function = float(ww80_semi_infinite(omega, g, mu0))
display(t.readable_table(pd.DataFrame({"quantity": ["wavelength_um", "omega", "g", "mu0", "g_star", "omega_star", "a", "b", "xi", "P", "manual", "function"],
                      "value": [optics["wavelength_um"][wi], omega, g, mu0, g_star, omega_star, a, b, xi, p, alpha_manual, alpha_function]}).rename(columns={"quantity":"中间量", "value":"数值"}), precision=6))
np.testing.assert_allclose(alpha_manual, alpha_function, atol=1e-12, rtol=0)
print(f"单波长手算：{optics['wavelength_um'][wi]:.3f} μm，反照率 {alpha_manual:.4f}")

## 从单波长扩展到一条光谱

现在只扩展波长轴：`ss_alb[ri, :, 0]` 是 480 个波长，半径、角度和光学选项仍固定，输出也应为 480 个数。左图给出散射输入，右图给出辐射传输输出。先观察 1 μm 附近的吸收变化，再与反照率曲线联系。这里的曲线直接从输入参数计算，不读取历史反照率数组。

In [ ]:
spectrum = ww80_semi_infinite(optics["ss_alb"][ri, :, 0], optics["asm_prm"][ri, :, 0], mu0)
assert spectrum.shape == (480,) and np.isfinite(spectrum).all()
assert abs(spectrum[wi] - alpha_manual) < 1e-12
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(optics["wavelength_um"], optics["ss_alb"][ri, :, 0], label=t.label("单次散射反照率 ω", "Single-scattering albedo ω"))
axes[0].plot(optics["wavelength_um"], optics["asm_prm"][ri, :, 0], label=t.label("不对称因子 g", "Asymmetry g"))
axes[1].plot(optics["wavelength_um"], spectrum, label="NumPy WW80", color=t.COLORS[0])
axes[0].set(xlim=(.3, 2.5), ylim=(0, 1.02), ylabel=t.label("散射参数（无量纲）", "Optical parameters (dimensionless)"))
axes[1].set(xlim=(.3, 2.5), ylim=(0, 1.02), ylabel=t.label("反照率（无量纲）", "Albedo (dimensionless)"))
for ax in axes: ax.set_xlabel(t.label("波长 / μm", "Wavelength / μm")); ax.legend(fontsize=8)
fig.tight_layout(); display(fig); plt.close(fig)

## 完整网格怎样排轴

批量计算扩展半径、角度和冰光学选项，采用 `(radius, wavelength, angle, ice_option)` 的顺序。`[:, :, None, :]` 给光学属性插入角度轴；角度余弦用 `(1, 1, 3, 1)` 广播，不必写 148×3×2 个循环。角度先转 float64 再进入 `deg2rad`。

共享入口 `load_forward` 读取散射参数，现算 **148×480×3×2＝426,240** 个 WW80 值。太阳通量为独立的小型输入 `solar_illumination.npz`；通量按波长加权后得到总反照率。下面先检查数组轴、有限值与能量范围，再用独立写法 `np.average` 核对加权。历史 MATLAB 对照放在后面单独计算。

In [ ]:
forward = t.load_forward()
recomputed = forward["ww_albedo"]
assert recomputed.shape == (148, 480, 3, 2) and recomputed.size == 426240
assert np.isfinite(recomputed).all() and 0 <= recomputed.min() <= recomputed.max() <= 1
np.testing.assert_allclose(recomputed[ri, :, 1, 0], spectrum, atol=1e-12)
# solar_flux 的轴为 (angle, wavelength)，逐角度独立加权。
broadband_check = np.stack([np.average(recomputed[:, :, ai, :], axis=1, weights=forward["solar_flux"][ai]) for ai in range(3)], axis=1)
np.testing.assert_allclose(broadband_check, forward["broadband_ww"], atol=1e-12)
print(f"NumPy 正演：{recomputed.size:,} 个值；全部有限且在 [0,1]，太阳谱加权已核对")

## 用图检验最初的预测

代表半径取 50、200、500、1000 μm。50 μm 和 1000 μm 分别用于说明较细新雪与较粗老雪的典型差别；实际雪况会形成连续的粒径分布。图中两角度使用同一坐标和颜色，先看 0.5 μm，再看 1.03、1.6 μm 附近。吸收较强的近红外和短波红外通常更能区分粒径。

这两张物理图及下一页 Fig. 3 都使用现算的 WW80 结果。保持同一正演分支后，粒径和角度的作用更容易连续解释。SNICAR-AD 单独用于后面的独立参考比较。

In [ ]:
figure = t.plot_optical_spectra(forward)
display(figure); plt.close(figure)

粒径效应在各波段大小不同，可见光区的差异较小。单用可见光波段估计粒径时，覆盖度与照明差异容易主导观测变化；近红外吸收特征提供了进一步区分它们的依据。为了概括总反射能量，还需按入射太阳谱加权：
$$\overline\alpha=\frac{\sum_c F_{\mathrm{sun},c}\alpha_c}{\sum_cF_{\mathrm{sun},c}}.$$
输入光谱是无量纲，通量在分子分母中约去，输出仍为无量纲。左图回答粒径和冰光学选项对宽波段量的影响；右图单独给出 Warren–Brandt 2008 相对 Warren 1984 的差值，单位为反照率百分点。

In [ ]:
figure = t.plot_broadband_albedo(forward)
display(figure); plt.close(figure)
display(t.readable_table(t.optics_table(forward).query("radius_um in [50, 200, 500, 1000] and solar_zenith_deg in [0, 60]"), {"radius_um":"半径 / μm", "solar_zenith_deg":"太阳角 / °", "ice_option":"冰选项", "ww80_albedo":"WW80加权反照率"}, precision=5))

## 单独核对 MATLAB 参考实现

前面的计算和制图只依赖散射参数与太阳谱。现在读取约 183 KB 的 `forward_reference.npz`：保留四种代表半径、480 波长、3 角度、2 光学选项的独立 MATLAB WW80 与 SNICAR-AD 输出。先比较同一 WW80 公式的 **11,520 个参考值**，再比较全部半径的太阳谱加权值；最大绝对差应小于 $10^{-6}$。

SNICAR-AD 使用 adding-doubling，与 WW80 的模型差值应另行解释。完整版历史 MAT 已移出教学仓库；此前全 426,240 值校核的最大差为 $5.16\times10^{-8}$，记录在来源说明。若显式生成 `data/cache/forward_models.mat`，下面同一个参考 cell 会追加全网格校核；没有该文件也能完成默认教学。

In [ ]:
reference = t.load_forward_reference()
indices = [int(np.flatnonzero(forward["radii_um"] == radius)[0]) for radius in reference["radii_um"]]
reference_error = float(np.max(abs(recomputed[indices] - reference["ww_albedo"])))
broadband_error = float(np.max(abs(forward["broadband_ww"] - reference["broadband_ww"])))
assert reference_error < 1e-6 and broadband_error < 1e-6
print(f"独立 WW80 参考：{reference['ww_albedo'].size:,} 个值；最大差 {reference_error:.3e}；加权最大差 {broadband_error:.3e}")
fig, ax = plt.subplots(figsize=(8.5, 3.8))
for ai, angle in enumerate(forward["angles_deg"][:2]):
    ax.plot(forward["radii_um"], 100 * (forward["broadband_ww"][:, ai, 0] - reference["broadband_snicar"][:, ai, 0]), label=f"{angle:g}°")
ax.set(xlabel=t.label("雪粒半径 / μm", "Snow radius / μm"), ylabel=t.label("WW80 − SNICAR-AD / 反照率百分点", "WW80 − SNICAR-AD / albedo pp"))
ax.axhline(0, color=".5", lw=.8); ax.legend(); fig.tight_layout(); display(fig); plt.close(fig)
# 可选完整对照只在此处读取，永远不进入主正演或合成真值输入。
full_reference_path = t.project_root() / "data/cache/forward_models.mat"
if full_reference_path.is_file():
    from scipy.io import loadmat
    full_reference = loadmat(full_reference_path, variable_names=["ww_albedo"])["ww_albedo"]
    assert full_reference.shape == recomputed.shape
    full_error = float(np.max(abs(recomputed - full_reference)))
    assert full_error < 1e-6
    print(f"可选完整 WW80 参考：426,240 个值；最大差 {full_error:.3e}")

## 自己改一次，再解释

前面的断言用于固定基线。下面是独立练习，改 `exercise_radius_um` 或 `exercise_angle_deg` 后，不要求曲线保持基线数值。先预测 200→1000 μm 对可见光和 1.025 μm 的影响，再比较打印结果。角度增大时光路与相函数的作用也会改变反照率，必须把角度保持一致后再比较粒径。

In [ ]:
exercise_radius_um = 1000  # 在 30:10:1500 μm 网格内选择。
exercise_angle_deg = 60.0  # 在 [0, 90]° 内选择。
exercise_ri = int(np.flatnonzero(optics["radii_um"] == exercise_radius_um)[0])
exercise_mu = np.cos(np.deg2rad(float(exercise_angle_deg)))
exercise_spectrum = ww80_semi_infinite(optics["ss_alb"][exercise_ri, :, 0], optics["asm_prm"][exercise_ri, :, 0], exercise_mu)
for wavelength in (.5, 1.025, 1.6):
    index = int(np.argmin(abs(optics["wavelength_um"] - wavelength)))
    print(f"{optics['wavelength_um'][index]:.3f} μm: 基线 {spectrum[index]:.4f} → 练习 {exercise_spectrum[index]:.4f}")

同一个雪面积比例可以对应多条不同粒径的雪谱。下一页固定混合比例的真值，检查把这些谱都近似为 200 μm 时，最小二乘会怎样改变雪系数。

下一步：[02 从三波段手算到 Fig. 3](02_fig3_sma.ipynb)。物理来源为 Wiscombe–Warren (1980)、SNICAR-ADv3 (Flanner et al., 2021) 及随仓库保留的 SPIRES 参考实现；散射参数子集从官方公开库提取并标明精度转换。